# 10 · Explanation Stability

## What this notebook covers
Feature importance is a key tool for model auditing, debugging, and regulatory compliance. But importance estimates are noisy — run the same permutation importance twice and you may get different rankings. This notebook quantifies explanation stability and shows how to detect features whose importance estimates are too noisy to trust.

## Why stability matters
A feature ranked #1 in one run and #7 in another is not reliably "important" — it is measurement noise. Presenting unstable importances to stakeholders is misleading. Stability analysis should be a standard part of any explanation workflow, especially when:
- Importances inform regulatory filings or audits
- Feature selection is based on importance ranking
- You're comparing importances before/after retraining

## Stability metrics implemented
| Metric | Measures |
|--------|----------|
| **Rank stability** (Spearman ρ) | Whether the importance *ordering* is consistent |
| **Top-k stability** (Jaccard) | Whether the same features appear in the top-k |
| **Sign stability** | Whether importance direction (positive/negative) is consistent |
| **CV per feature** | Feature-level coefficient of variation across bootstrap runs |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

X, y = make_classification(
    n_samples=2000, n_features=15, n_informative=8, n_redundant=3,
    n_repeated=0, random_state=0
)
feature_names = [f"feat_{i:02d}" for i in range(X.shape[1])]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

clf = GradientBoostingClassifier(n_estimators=100, random_state=0)
clf.fit(X_tr, y_tr)
print(f"Test AUC: {roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1]):.4f}")

In [ ]:
def get_permutation_importance(clf, X, y, n_repeats=10, random_state=0):
    """Run permutation importance and return mean importances array."""
    result = permutation_importance(clf, X, y, n_repeats=n_repeats,
                                    random_state=random_state, scoring="roc_auc")
    return result.importances_mean

def bootstrap_importance_runs(clf, X_tr, y_tr, X_te, y_te, feature_names,
                               n_bootstrap=30, n_repeats=5):
    """
    Re-train clf on bootstrap samples of training data,
    evaluate permutation importance on fixed test set.
    Returns matrix (n_bootstrap x n_features).
    """
    n = len(X_tr)
    importance_matrix = []
    for b in range(n_bootstrap):
        idx = np.random.randint(0, n, n)
        clf_b = GradientBoostingClassifier(n_estimators=100, random_state=b)
        clf_b.fit(X_tr[idx], y_tr[idx])
        imp = get_permutation_importance(clf_b, X_te, y_te, n_repeats=n_repeats, random_state=b)
        importance_matrix.append(imp)
    return np.array(importance_matrix)  # (n_bootstrap, n_features)

print("Running bootstrap importance (30 runs × 5 repeats per run)...")
imp_matrix = bootstrap_importance_runs(clf, X_tr, y_tr, X_te, y_te, feature_names, n_bootstrap=30)
print(f"Importance matrix shape: {imp_matrix.shape}")

In [ ]:
def rank_stability(imp_matrix):
    """Mean Spearman ρ across all pairs of bootstrap runs."""
    n = imp_matrix.shape[0]
    rhos = []
    for i in range(n):
        for j in range(i+1, n):
            rho, _ = stats.spearmanr(imp_matrix[i], imp_matrix[j])
            rhos.append(rho)
    return np.mean(rhos)

def top_k_stability(imp_matrix, k=5):
    """Mean Jaccard similarity of top-k sets across all pairs."""
    n = imp_matrix.shape[0]
    jaccards = []
    for i in range(n):
        top_i = set(np.argsort(imp_matrix[i])[-k:])
        for j in range(i+1, n):
            top_j = set(np.argsort(imp_matrix[j])[-k:])
            jaccards.append(len(top_i & top_j) / len(top_i | top_j))
    return np.mean(jaccards)

def sign_stability(imp_matrix):
    """Fraction of features that have the same sign in >90% of runs."""
    signs = np.sign(imp_matrix)
    dominant_sign_frac = (np.abs(signs.sum(axis=0)) / signs.shape[0])
    return dominant_sign_frac.mean()

print(f"Rank stability (mean Spearman ρ) : {rank_stability(imp_matrix):.4f}")
print(f"Top-5 stability (mean Jaccard)   : {top_k_stability(imp_matrix, k=5):.4f}")
print(f"Sign stability (mean)            : {sign_stability(imp_matrix):.4f}")

In [ ]:
# Per-feature CV (coefficient of variation) — high CV = unstable
means = imp_matrix.mean(axis=0)
stds  = imp_matrix.std(axis=0)
cv    = stds / (np.abs(means) + 1e-8)

stability_df = pd.DataFrame({
    "feature" : feature_names,
    "mean_imp": means.round(5),
    "std_imp" : stds.round(5),
    "cv"      : cv.round(3),
    "shaky"   : cv > 0.5,
}).sort_values("mean_imp", ascending=False)

print(stability_df.to_string(index=False))
print(f"\nShaky features (CV > 0.5): {stability_df['shaky'].sum()}/{len(stability_df)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Feature importances with error bars
idx = np.argsort(means)
axes[0].barh(
    [feature_names[i] for i in idx],
    means[idx],
    xerr=stds[idx],
    color=["tomato" if cv[i] > 0.5 else "steelblue" for i in idx],
    alpha=0.8
)
axes[0].set_title("Bootstrap permutation importances (± 1 SD)
Red = high CV (shaky)")
axes[0].set_xlabel("Importance (ΔAUC)")

# CV per feature
axes[1].bar(feature_names, cv, color=["tomato" if c > 0.5 else "steelblue" for c in cv], alpha=0.8)
axes[1].axhline(0.5, color="black", linestyle="--", label="CV=0.5 threshold")
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_title("Coefficient of Variation per feature")
axes[1].set_ylabel("CV")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{base}/10_explanation_stability.png", dpi=120)
plt.show()